# 🐾 AnimalMind — ViT Dog Breed Classifier Training Pipeline (>90% Target Accuracy)

This notebook trains and calibrates the ViT dog breed classifier on **Stanford Dogs (120 breeds)** with:
- **Data Augmentations**: RandAugment(num_ops=2, magnitude=9), ColorJitter, MixUp, CutMix.
- **Regularization**: Label Smoothing (0.1), EMA, Stochastic Depth (0.2).
- **Uncertainty Calibration**: Temperature Scaling via L-BFGS to minimize Expected Calibration Error (ECE).
- **Model Export**: Automatic upload to Hugging Face Hub (`firstoff/animalmind-breed-classifier`).

In [1]:
# 1. Verify GPU Acceleration
!nvidia-smi

Mon Jul 27 12:08:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Clone Repository & Install Training Dependencies
!git clone https://github.com/firstoff23/AnimalMind.git
%cd AnimalMind/ml_backend
!pip install -q -r requirements_training.txt

Cloning into 'AnimalMind'...
remote: Enumerating objects: 47421, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 47421 (delta 158), reused 203 (delta 132), pack-reused 47171 (from 2)
Receiving objects: 100% (47421/47421), 100.76 MiB | 12.72 MiB/s, done.
Resolving deltas: 100% (17879/17879), done.
/content/AnimalMind/ml_backend


In [3]:
# 3. Configure Hugging Face Token (Optional - for direct HF Hub upload)
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("🔑 Hugging Face Token loaded successfully.")
except Exception:
    print("ℹ️ No Colab Secret 'HF_TOKEN' found. Model will be saved locally.")

ℹ️ No Colab Secret 'HF_TOKEN' found. Model will be saved locally.


In [4]:
# 4. Launch 50-Epoch GPU Training Pipeline
!python -m training.train_dog_breeds \
    --epochs 50 \
    --batch-size 32 \
    --lr 3e-4 \
    --model-name google/vit-base-patch16-224 \
    --output-dir models/animalmind-breed-classifier \
    --push-to-hub firstoff/animalmind-breed-classifier

 AnimalMind — Fine-Tuning ViT Dog Breed Classifier Pipeline (>90% target)
Model Backbone  : google/vit-base-patch16-224
Batch Size      : 32
Epochs          : 50
Learning Rate   : 0.0003
Dry Run         : False
[Device] Running on: cuda
[Dataset] Loading Stanford Dogs dataset from Hugging Face Datasets...
[Dataset] Warning: Could not load Stanford Dogs remotely (No Stanford Dogs dataset repository accessible.). Using synthetic fallback.
config.json: 100% 69.7k/69.7k [00:00<00:00, 81.5MB/s]
[transformers] You passed `num_labels=120` which is incompatible to the `id2label` map of length `1000`.

model.safetensors: downloading bytes:  53% 182M/346M [00:01<00:00, 250MB/s, 14.2MB/s  ]
model.safetensors: downloading bytes:  79% 275M/346M [00:01<00:00, 284MB/s, 22.7MB/s  ]
model.safetensors: downloading bytes:  90% 310M/346M [00:01<00:00, 199MB/s, 27.9MB/s  ]
model.safetensors: reconstructing file:  77% 268M/346M [00:01<00:00, 183MB/s, 12.8MB/s  ] 
model.safetensors: downloading bytes: 100% 3

In [5]:
# 5. Display Calibration & Metrics Report
import json
import torch

metrics_path = "training/training_metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"🏆 Best Validation Accuracy : {data.get('best_val_accuracy', 0)*100:.2f}%")
print(f"🌡️ Calibrated Temperature T : {data.get('calibrated_temperature', 1.0):.4f}")

temp_data = torch.load("models/temperature.pt")
print("Saved Temperature Tensor:", temp_data)

🏆 Best Validation Accuracy : 3.12%
🌡️ Calibrated Temperature T : 1.8144
Saved Temperature Tensor: {'temperature': 1.814359188079834}
